#  **Unstructured Partitioning** 



---

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

---

## **파티셔닝(Partitioning) 개념**

- **파티셔닝**은 비정형 문서를 **Title**, **NarrativeText**, **ListItem** 등 구조화된 요소로 분할하는 과정

- 문서의 각 부분을 **의미 있는 단위**로 구분하여 데이터 활용도 향상

- 사용자가 필요한 콘텐츠만 **선택적으로 추출** 가능

---

## **문서별 파티셔닝 전략**

### 1. **PDF/이미지 문서**

`(1) "auto"`

- **auto 전략**은 문서 특성에 따라 최적의 파티셔닝 방법을 자동 선택 (기본 전략)
- 텍스트 추출 가능성에 따라 **fast** 또는 **ocr_only** 전략으로 자동 전환

In [ ]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="data/transformer.pdf",   
    strategy="auto", 
)

len(elements)

In [ ]:
def count_elements(elements):
    """문서 구성 요소별 개수 확인"""

    elements_by_type = {}

    for element in elements:
        if "Title" in str(type(element)):
            if "Title" not in elements_by_type:
                elements_by_type["Title"] = []
            elements_by_type["Title"].append(element)

        elif "Table" in str(type(element)):
            if "Table" not in elements_by_type:
                elements_by_type["Table"] = []
            elements_by_type["Table"].append(element)

        elif "Text" in str(type(element)):
            if "Text" not in elements_by_type:
                elements_by_type["Text"] = []
            elements_by_type["Text"].append(element)
        else:
            if "Other" not in elements_by_type:
                elements_by_type["Other"] = []
            elements_by_type["Other"].append(element)

    return elements_by_type


# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Ohter 타입의 요소 확인
for element in elements_by_type["Other"]:
    print(type(element))

### **[실습]**

- 종류별로 1개씩 요소를 선택하여 내용과 구조를 분석합니다. 

In [ ]:
# Text 타입의 요소 확인
text_el = elements_by_type["Text"][10]

# 여기에 코드를 추가하세요.

In [ ]:
# Title 타입의 요소 확인
title_el = elements_by_type["Title"][10]

# 여기에 코드를 추가하세요.

In [ ]:
# Other 타입의 요소 확인
other_el = elements_by_type["Other"][0]

# 여기에 코드를 추가하세요.

`(2) "fast" `

- **PDF 텍스트 추출**을 위해 pdfminer 라이브러리를 사용
- 텍스트 추출이 가능한 문서에서 **최고 속도**의 처리를 보장

In [ ]:
from unstructured.partition.pdf import partition_pdf

fast_elements = partition_pdf(
    filename="data/transformer.pdf",  # PDF 파일 경로 지정
    strategy="fast",                  # 파티셔닝 전략 설정 
    include_page_breaks=True          # 페이지 구분자(PageBreak 요소)를 포함
)

# 문서 구성 요소 개수 확인
len(fast_elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(fast_elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Ohter 타입의 요소 확인
for element in elements_by_type["Other"]:
    print(type(element))

In [ ]:
# page break 포함해서 일부만 확인
for element in fast_elements[:25]:
    if element.category == "PageBreak":
        print("-- Page Break --")
        print(element.text)
        
    else:
        print(element.text)

### **[실습]**

- Footer를 제외한 나머지 요소는 원래 순서를 유지한 채로 재구조화합니다. (Footer 요소만 제거) 

In [ ]:
# 여기에 코드를 작성하세요.

In [ ]:
# 문서 요소를 출력 
for element in cleaned_elements[:30]:
    print(element.text)

`(3) "hi_res" `

- **문서 레이아웃 분석**을 위해 객체 탐지 모델을 활용
- 높은 정밀도가 필요한 문서 요소 **분류 작업**에 적합
- 레이아웃 기반으로 **문서 구조 정보**를 상세하게 추출할 수 있음 

In [ ]:
from unstructured.partition.pdf import partition_pdf

# PDF 문서를 고해상도 전략으로 파티셔닝하고 이미지/테이블을 추출
hi_res_elements = partition_pdf(
    # 기본 설정
    filename="data/transformer.pdf",    # 처리할 PDF 파일의 경로
    strategy="hi_res",                  # 고해상도 파티셔닝 전략 사용
    hi_res_model_name="yolox",  # 객체탐지 모델 지정 
    
    # 구조 분석 설정
    infer_table_structure=True,         # 표 구조를 자동으로 추론하여 HTML 형식으로 변환
    languages=["eng", "kor"],           # 영어와 한국어 문서 처리 지원
    
    # 이미지 추출 설정
    extract_images_in_pdf=True,         # PDF 내 이미지 추출 활성화
    extract_image_block_types=[         # 추출할 요소 유형 지정
        "Image",                        # 일반 이미지
        "Table"                         # 표 이미지
    ],
    extract_image_block_output_dir="output_images"  # 추출된 이미지 저장 경로
)

# 문서 구성 요소 개수 확인
len(hi_res_elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(hi_res_elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# 테이블 요소 확인
table_elements = elements_by_type["Table"]
table_elements[0]

In [ ]:
# 테이블 요소의 속성 확인
table_elements[0].to_dict()

In [ ]:
# 테이블 요소의 HTML 텍스트 확인
table_elements[0].metadata.text_as_html

In [ ]:
# 판다스 데이터프레임으로 변환
df = pd.read_html(table_elements[0].metadata.text_as_html)[0]
df

In [ ]:
# 마크다운 형식으로 변환
print(df.to_markdown())

### **[실습]**

- 두 번째 테이블 요소를 마크다운과 데이터프레임 형식으로 변환합니다. 
- 변환된 데이터를 LLM에 전달하여 내용을 요약하고, 그 결과를 비교합니다. 

In [ ]:
# 두 번째 테이블 요소 확인
table_elements[1].to_dict()

In [ ]:
# 두 번째 테이블 요소를 데이터프레임으로 변환


In [ ]:
# 마크다운 형식으로 변환


In [ ]:
# LLM에 각 데이터를 전달하여 내용 요약 (마크다운 형식)
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4.1-mini")   

# 프롬프트 템플릿 정의
prompt_template = PromptTemplate(
    input_variables=["table"],
    template="Summarize the table:\n\n[Table]\n{table}\n\n[Summary (in 한국어)]\n"
)
# 프롬프트 템플릿 생성
prompt = prompt_template.format(table=df2.to_markdown())

# LLM에 프롬프트 전달하여 요약 생성
summary = llm.invoke(prompt)

# 요약 결과 출력
pprint(summary.content)

In [ ]:
# 데이터프레임을 문자열로 변환하여 출력
print(str(df2))

In [ ]:
# LLM에 각 데이터를 전달하여 내용 요약 (데이터프레임 형식)
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4.1-mini")

# 프롬프트 템플릿 정의
prompt_template = PromptTemplate(
    input_variables=["table"],
    template="Summarize the table:\n\n[Table]\n{table}\n\n[Summary (in 한국어)]\n"
)

# 프롬프트 템플릿 생성
prompt = prompt_template.format(table=str(df2))

# LLM에 프롬프트 전달하여 요약 생성
summary = llm.invoke(prompt)

# 요약 결과 출력
pprint(summary.content)

`(4) "ocr_only"`

- **Tesseract OCR**을 활용한 텍스트 추출 방식을 채택
- **다중 열 구조**의 복잡한 문서에서 hi_res의 대안으로 사용
- 이미지 기반 문서의 **텍스트 인식**에 특화

In [ ]:
from unstructured.partition.pdf import partition_pdf

ocr_elements = partition_pdf(
    filename="data/transformer.pdf",     
    strategy="ocr_only",                 
    languages=["eng", "kor"],            # 문서에 포함된 언어를 지정 (메타데이터에 기록)
    max_partition=1500                   # 최대 텍스트 분할 크기(문자 수) 
)

# 문서 구성 요소 개수 확인
len(ocr_elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(ocr_elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Ohter 타입의 요소 확인
for element in elements_by_type["Other"]:
    print(type(element))

### **[실습]**

- 다음 pdf 문서를 hi_res와 ocr_only 전략으로 변환합니다. 
    - data/리비안_KR_with_table.pdf
    
- 테이블 부분을 찾아서 내용과 구조를 확인합니다. 

In [ ]:
# 문서(data/리비안_KR_with_table.pdf) 파티셔닝 - OCR 전략
test_ocr_elements = None

# 문서 구성 요소 개수 확인
len(test_ocr_elements)


# 문서 구성 요소별 개수 확인


# Ohter 타입의 요소 확인


In [ ]:
# 테이블 위치의 요소 확인 - 텍스트에 "순이익 (백만 USD)" 포함

# 테이블 요소의 속성 확인
test_ocr_el.to_dict()

In [ ]:
# 문서(data/리비안_KR_with_table.pdf) 파티셔닝 - high_res 전략
test_hi_res_elements = None

# 문서 구성 요소 개수 확인
len(test_hi_res_elements)

# 문서 구성 요소별 개수 확인


# Ohter 타입의 요소 확인


In [ ]:
# 테이블 위치의 요소 확인

# 테이블 요소의 속성 확인
test_hi_res_el.to_dict()
        

In [ ]:
# 테이블 요소의 HTML 텍스트 확인
test_hi_res_el.metadata.text_as_html

In [ ]:
# 판다스 데이터프레임으로 변환
df = pd.read_html(test_hi_res_el.metadata.text_as_html)[0]
df

### 2. **HTML**

- **웹 문서 구조**를 분석하여 HTML 요소를 체계적으로 추출
- **URL 기반**으로 웹 페이지를 직접 처리 가능

`(1) "로컬 파일"`

- **partition_html** 함수는 로컬에 저장된 HTML 파일을 읽어 **문서 요소로 분할**

- `filename` 매개변수를 통해 **로컬 HTML 파일의 경로**를 지정 가능

In [ ]:
from unstructured.partition.html import partition_html

# 로컬 파일
elements = partition_html(filename="data/example.html")

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
# 문서 구성 요소 확인
for element in elements:
    print(element.to_dict())

`(2) "URL에서 직접 파티셔닝"`

- **URL을 직접 지정**하여 웹페이지의 HTML 콘텐츠를 파티셔닝 가능

- **사용자 정의 헤더**를 설정 및 `User-Agent` 설정 지원

In [ ]:
# URL에서 직접 파티셔닝
elements = partition_html(
    url="https://example.com",
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0"
    } # 사용자 정의 헤더 지정
)

# 문서 구성 요소 개수 확인
len(elements)

`(3) "구성 요소의 계층적 관계 처리"`

- **Unstructured**로 추출한 문서 요소들의 **계층적 구조**를 재구성하여 사용

- 각 요소는 고유한 **element_id**를 가지며, `parent_id`를 통해 상위 요소와의 관계가 정의됨 

- **재귀적 처리** 방식을 통해 부모-자식 관계를 트리 구조로 재구성 가능 

In [ ]:
# URL에서 직접 파티셔닝
web_elements = partition_html(
    url="https://python.langchain.com/docs/how_to/structured_output/#the-with_structured_output-method",
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0"
    } # 사용자 정의 헤더 지정
)

# 문서 구성 요소 개수 확인
len(web_elements)

In [ ]:
# 문서 구성 요소 확인
for element in web_elements:
    print(element.to_dict())

In [ ]:
def build_hierarchy(elements):
    """  문서 요소의 계층 구조를 구축하는 함수
    Args:
        elements (list): 문서 요소 리스트
    Returns:
        list: 계층 구조가 적용된 문서 요소 리스트
    """

    # 딕셔너리 초기화
    elements_by_id = {}
    children_by_id = {}
    root_elements = []
    
    # 1 단계 - 모든 요소를 딕셔너리에 추가
    for element in elements:
        element_dict = element.to_dict()
        element_id = element_dict['element_id']
        elements_by_id[element_id] = element_dict
        children_by_id[element_id] = []

    # 2 단계 - 부모-자식 관계 구축
    for element in elements:
        element_dict = element.to_dict()
        parent_id = element_dict['metadata'].get('parent_id')
        
        if parent_id:
            # 부모가 있는 경우 - 자식 요소로 추가
            if parent_id in children_by_id:
                children_by_id[parent_id].append(element_dict)
        else:
            # 부모가 없는 경우 - 루트 요소로 추가
            root_elements.append(element_dict)
    
    # 3 단계 - 자식 요소 추가
    def add_children(element_dict):
        element_id = element_dict['element_id']
        if children_by_id[element_id]:
            element_dict['children'] = [
                add_children(child) for child in children_by_id[element_id]
            ]
        return element_dict
    
    # 모든 루트 요소에 대해 자식 요소 추가
    hierarchy = [add_children(root) for root in root_elements]
    
    return hierarchy

# 문서 요소의 계층 구조 구축
doc_hierarchy = build_hierarchy(web_elements)

# 계층 구조 확인
pprint(doc_hierarchy)

In [ ]:
# 계층적 구조를 가진 문서를 시각화 
def print_hierarchy(elements, level=0):
    for element in elements:
        indent = "  " * level
        print(f"{indent}- {element['type']}: {element['text'][:50]}...")
        if 'children' in element:
            print_hierarchy(element['children'], level + 1)
            
print_hierarchy(doc_hierarchy)

`(4) "LangChain Document 변환하여 계층적 검색 처리"`



In [ ]:
from langchain_core.documents import Document

def convert_hierarchy_to_documents(elements):
    """계층적 구조를 가진 문서를 LangChain Document 객체로 변환하는 함수
    Args:
        elements (list): 계층적 구조를 가진 문서 요소 리스트
    Returns:
        list: LangChain Document 객체 리스트
    """
    
    documents = []
    
    for element in elements:
        element_dict = element.to_dict()
        # 메타데이터 구성
        metadata = {
            'element_id': element_dict.get('element_id'),
            'type': element_dict.get('type', ''),
            'parent_id': element_dict.get('metadata', {}).get('parent_id', ''),
            'url': element_dict.get('metadata', {}).get('url'), 
        }

        # Document 객체 생성
        doc = Document(page_content=element.text, metadata=metadata)
        documents.append(doc)
    
    return documents

# 계층적 구조를 가진 문서를 LangChain Document 객체로 변환
web_documents = convert_hierarchy_to_documents(web_elements)

# 변환된 Document 객체 확인
for doc in web_documents[:5]:
    print(doc)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# OpenAI 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터스토어 생성
vectorstore = Chroma.from_documents(
    web_documents,
    embeddings,
    collection_name="web_elements",
    persist_directory="./chroma_db" 
)

# 벡터스토어에 저장된 문서 개수 확인
print(vectorstore._collection.count())

In [ ]:
# 벡터스토어에 저장된 문서 확인
vectorstore.similarity_search(
    query="What is the tool calling?",
    k=3
)

In [ ]:
from typing import List, Any
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStoreRetriever

class CustomRetriever(VectorStoreRetriever):
    """ 사용자 쿼리에 대한 검색을 먼저 처리하고, 그 결과를 기반으로 문서를 검색하는 리트리버
    검색 결과의 메타데이터 속성의 parent_id를 사용하여 문서 검색
    parent_id를 메타데이터로 가지고 있는 문서를 추가해서 반환 
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.search_kwargs = kwargs.get("search_kwargs", {})
        self.search_type = kwargs.get("search_type", "similarity")
        self._metadatas = self.vectorstore.get()['metadatas']
        self._doc_ids = self.vectorstore.get()['ids']

        
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun, **kwargs: Any
    ) -> list[Document]:
        _kwargs = self.search_kwargs | kwargs
        if self.search_type == "similarity":
            docs = self.vectorstore.similarity_search(query, **_kwargs)
            docs = self._get_additional_docs(docs)
        elif self.search_type == "similarity_score_threshold":
            docs_and_similarities = (
                self.vectorstore.similarity_search_with_relevance_scores(
                    query, **_kwargs
                )
            )
            docs = [doc for doc, _ in docs_and_similarities]
        elif self.search_type == "mmr":
            docs = self.vectorstore.max_marginal_relevance_search(query, **_kwargs)
        else:
            msg = f"search_type of {self.search_type} not allowed."
            raise ValueError(msg)
        return docs


    def _get_additional_docs(self, docs: List[Document]) -> List[Document]:
        """추가 문서 검색"""
        additional_docs = []
        for doc in docs:
            parent_id = doc.metadata.get('parent_id')
            if len(parent_id) > 0:
                # parent_id와 일치하는 문서들 추가
                _same_parent_ids = [i for i, m in zip(self._doc_ids, self._metadatas) if m['parent_id'] == parent_id]
                _added_docs = self.vectorstore.get_by_ids(_same_parent_ids)
                additional_docs.extend(_added_docs)

        # 중복된 문서 제거
        unique_additional_docs = []
        for doc in docs + additional_docs:
            if doc not in unique_additional_docs:
                unique_additional_docs.append(doc)

        print(f"최종 문서 개수: {len(unique_additional_docs)}")
        return unique_additional_docs
    

# 사용자 쿼리에 대한 검색을 먼저 처리하고, 그 결과를 기반으로 문서를 검색하는 리트리버
retriever = CustomRetriever(
    vectorstore=vectorstore,
    search_kwargs={"k": 3}
)

# 검색 실행 
search_results = retriever.invoke("What is the tool calling?")
 
for doc in search_results:
    print(doc)

### 3. **MS Office 문서 (DOCX, PPTX, XLSX)**

`(1) "DOCX 파일"`

- **MS Office 문서**를 partition_docx로 구조적으로 분석

- 문서의 **스타일 메타데이터**를 활용하여 요소를 분류
    - "Heading 1" → Title 요소
    - "Body Text" → NarrativeText 요소
    - "Normal" → NarrativeText 요소

- **헤더와 푸터** 등 문서의 구조적 요소를 정확하게 추출
    - Header 요소는 섹션 시작에 배치
    - Footer 요소는 섹션 끝에 배치

- **페이지 정보**를 포함한 문서의 상세 구조를 파악 가능
    - 페이지 번호가 metadata.page_number에 포함
    - Word 렌더러가 삽입한 페이지 나누기 감지
    - 사용자가 삽입한 페이지 나누기 감지

In [ ]:
from unstructured.partition.docx import partition_docx

# 파일명으로 처리
elements = partition_docx(filename="data/개인정보보호법.docx")

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Ohter 타입의 요소 확인
for element in elements_by_type["Other"]:
    print(type(element))

In [ ]:
# 문서 구성 요소 확인
for element in elements[:5]:
    print(element.to_dict())

In [ ]:
# 헤더/푸터 유형 확인
# element.metadata.header_footer_type (다음 중 하나의 값을 가짐)

# 1. "primary": 기본 헤더/푸터
# 2. "first_page": 첫 페이지 전용 헤더/푸터
# 3. "even_page": 짝수 페이지 전용 헤더/푸터

# 헤더만 추출
headers = [el for el in elements if el.category == "Header"]

# 헤더 개수 확인
print(f"Header Count: {len(headers)}")

# 헤더 유형별 분류
for header in headers:
    header_type = header.metadata.header_footer_type
    print(f"Type: {header_type}, Content: {header.text}")

In [ ]:
# 메타데이터 딕셔너리 반환
element.metadata.to_dict()  

- page_number: 페이지 번호
- category_depth: 요소 계층 구조 깊이
- coordinates: 요소 위치 정보
- emphasized_text_contents: 강조된 텍스트
- emphasized_text_tags: 강조 태그 정보

In [ ]:
# 특정 페이지의 요소만 추출
page_2_elements = [el for el in elements if el.metadata.page_number == 2]

for el in page_2_elements:
    print(el.to_dict())

`(2) "PPTX 파일"`

- **프레젠테이션 문서**를 partition_pptx로 처리
- **페이지 구분**을 포함한 상세 슬라이드 분석이 가능
- **슬라이드 요소**를 구조적으로 추출
    - Title: 슬라이드 제목
    - NarrativeText: 본문 텍스트
    - ListItem: 글머리 기호 목록
    - Table: 표 데이터
    - FigureCaption: 이미지 캡션

In [ ]:
from unstructured.partition.pptx import partition_pptx

# 파일명으로 처리
elements = partition_pptx(
    filename="data/한국철도공사_8대도시(목포)_관광 형태 분석 보고서_20220101.pptx",
    include_page_breaks=True  # 슬라이드 간 구분자 추가
    )

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Ohter 타입의 요소 확인
for element in elements_by_type["Other"]:
    print(type(element))

In [ ]:
# 슬라이드별 요소 분리
def get_slide_elements(elements, slide_number):
    return [el for el in elements 
            if el.metadata.page_number == slide_number]

# 슬라이드 1의 요소 확인
slide_1_elements = get_slide_elements(elements, 1)

for el in slide_1_elements:
    print(el.to_dict())

`(3) "XLSX 파일"`

- **Excel 문서**를 partition_xlsx로 구조화된 데이터로 변환
- 각 워크시트를 독립된 **테이블 객체**로 처리
- **HTML 테이블** 형식으로 데이터를 표현 가능

In [ ]:
from unstructured.partition.xlsx import partition_xlsx

# 파일명으로 처리
elements = partition_xlsx(
    filename="data/경기도교육청_행정구역별_학제별_학급_학생_교원_20240401.xlsx"
    )

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

In [ ]:
# Table 요소 확인
table_elements = elements_by_type["Table"]
table = table_elements[0]

# Table 요소의 텍스트 확인
pprint(table.text[:1000])

In [ ]:
pprint(table.metadata.text_as_html[:1000])

In [ ]:
import pandas as pd

# 판다스 데이터프레임으로 변환
df = pd.read_html(table.metadata.text_as_html)[0]
df.head()

In [ ]:
# 특정 시트 찾기
def find_sheet_by_name(elements, sheet_name):
    return next((el for el in elements 
                if el.metadata.page_name == sheet_name), None)

# 시트 이름으로 요소 찾기
sheet = find_sheet_by_name(elements, "요약정보")

# 시트 요소의 텍스트 확인
pprint(sheet.text[:1000])

### 4. **일반 텍스트 파일**

- **텍스트 문서**를 partition_text로 구조화된 요소로 분할
- **단락 그룹핑** 기능으로 끊어진 문단을 자동으로 재구성
- **파티션 크기**와 **인코딩** 설정으로 유연한 처리가 가능

In [ ]:
from unstructured.partition.text import partition_text

# 파일명으로 처리
elements = partition_text(filename="data/테슬라_KR.txt")

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
# 문서 구성 요소별 개수 확인
elements_by_type = count_elements(elements)

for key, value in elements_by_type.items():
    print(f"{key}: {len(value)}")

`(1) 단락 그룹핑`

- 줄바꿈으로 분리된 텍스트를 하나의 단락으로 병합
- 기본적으로 `\n`으로 분리된 줄 결합
- `\n\n`은 단락 구분자로 처리
- line_split, paragraph_split 매개변수로 구분자 커스터마이징 가능

In [ ]:
from unstructured.cleaners.core import group_broken_paragraphs

text = """첫 번째 줄: 여기에는 첫 번째 단락이 포함됩니다.
두 번째 줄: 여기에는 두 번째 단락이 포함됩니다.

새로운 단락의 내용이 여기서 시작됩니다. 이것은
첫 번째 줄이 아닙니다."""

elements = partition_text(
    text=text,
    paragraph_grouper=group_broken_paragraphs
)

# 문서 구성 요소 확인
for element in elements:
    print(element.to_dict())

`(2) 커스텀 단락 구분`

- 줄바꿈으로 분리된 텍스트를 하나의 단락으로 병합
- 기본적으로 `\n`으로 분리된 줄 결합
- `\n\n`은 단락 구분자로 처리
- line_split, paragraph_split 매개변수로 구분자 커스터마이징 가능

In [ ]:
import re
from unstructured.partition.text import partition_text
from unstructured.cleaners.core import group_broken_paragraphs

# 예시 텍스트
text = """첫 번째 단락이 시작되는 첫 번째 줄입니다.
계속되는 내용이 여기에 포함됩니다.


두 번째 단락이 시작되는 첫 번째 줄입니다.
계속되는 내용이 여기에 포함됩니다.



세 번째 단락이 시작되는 첫 번째 줄입니다.
마지막 내용이 여기에 포함됩니다."""

# 3줄 이상의 공백을 단락 구분자로 설정
paragraph_pattern = re.compile(r"(\s*\n\s*){3}")  

# 커스텀 단락 구분 적용
elements = partition_text(
    text=text,
    paragraph_grouper=lambda text: group_broken_paragraphs(
        text, 
        paragraph_split=paragraph_pattern
    )
)

# 문서 구성 요소 확인
for element in elements:
    print(element.to_dict())

### 5. **CSV/TSV 파일**

- **CSV/TSV 파일**을 단일 테이블 구조로 파싱
- partition_csv와 partition_tsv로 각 **파일 형식**에 맞는 처리를 수행
- **HTML 테이블** 형식으로 데이터를 시각화 가능

`(1) CSV 파일`

In [ ]:
from unstructured.partition.csv import partition_csv

# 파일명으로 처리
elements = partition_csv(filename="data/kbo_teams_2023.csv")

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
table = elements[0]

# 일반 텍스트로 출력
print(table.text)

# HTML 형식으로 출력
print(table.metadata.text_as_html)

In [ ]:
# 판다스 데이터프레임으로 변환
df = pd.read_html(table.metadata.text_as_html)[0]
df.head()

`(2) TSV 파일`

In [ ]:
from unstructured.partition.tsv import partition_tsv

# 파일명으로 처리
elements = partition_tsv(filename="data/kbo_teams_2023.tsv")

# 문서 구성 요소 개수 확인
len(elements)

In [ ]:
table = elements[0]

# 일반 텍스트로 출력
print(table.text)

# HTML 형식으로 출력
print(table.metadata.text_as_html)

In [ ]:
# 판다스 데이터프레임으로 변환
df = pd.read_html(table.metadata.text_as_html)[0]
df.head()

---

## **[실습] PDF 자동 문서 감지**

- pdf 문서에 테이블이 포함되어 있는지 partition_pdf(hi_res 전략)를 사용하여 탐지
- 탐지된 결과에 따라 각기 다른 텍스트 전처리 작업을 수행 
    - 테이블이 있는 경우: HTML 요소를 텍스트 요소와 함께 추출
    - 테이블이 없는 경우: 텍스트 요소만 추출 


In [ ]:
# 여기에 코드를 작성하세요.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import Table

# 상태 정의
class State(TypedDict):
    file_path: str  # 입력 PDF 파일 경로
    elements: list  # PDF 문서 요소들
    has_table: bool  # 테이블 존재 여부
    table_data: list  # 테이블 데이터 (있는 경우)
    content: str  # 일반 텍스트 내용

def check_table(state: State):
    """PDF 문서를 파싱하고 테이블 존재 여부 확인"""
    elements = partition_pdf(
        filename=state["file_path"],
        strategy="hi_res", # 고해상도 파티셔닝 전략 사용
        infer_table_structure=True, # 표 구조를 자동으로 추론하여 HTML 형식으로 변환
        languages=["eng", "kor"], # 영어와 한국어 문서 처리 지원
        extract_images_in_pdf=False, # PDF 내 이미지 추출 활성화
        )


    has_table = any(isinstance(element, Table) for element in elements)
    
    return {
        "elements": elements,
        "has_table": has_table
    }

def process_with_table(state: State):
    """테이블이 있는 경우 처리"""
    tables = []
    text_content = []
    
    for element in state["elements"]:
        if isinstance(element, Table):
            tables.append({
                "html": element.metadata.text_as_html,
                "text": str(element)
            })
        else:
            text_content.append(str(element))
    
    return {
        "table_data": tables,
        "content": "\n".join(text_content)
    }

def process_without_table(state: State):
    """테이블이 없는 경우 처리"""
    text_content = [str(element) for element in state["elements"]]
    
    return {
        "content": "\n".join(text_content)
    }

def router(state: State) -> Literal["with_table", "without_table"]:
    """테이블 유무에 따른 라우팅"""
    return "with_table" if state["has_table"] else "without_table"

# 그래프 구성
workflow = StateGraph(State)

# 노드 추가
workflow.add_node("check_table", check_table)
workflow.add_node("process_with_table", process_with_table)
workflow.add_node("process_without_table", process_without_table)

# 엣지 추가
workflow.add_edge(START, "check_table")

# 조건부 엣지 추가
workflow.add_conditional_edges(
    "check_table",
    router,
    {
        "with_table": "process_with_table",
        "without_table": "process_without_table"
    }
)

workflow.add_edge("process_with_table", END)
workflow.add_edge("process_without_table", END)

# 그래프 컴파일
graph = workflow.compile()

# 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))  

In [ ]:
# 테이블이 있는 PDF 파일 경로
file_path_with_table = "data/리비안_KR_with_table.pdf"

# 그래프 실행
result_with_table = graph.invoke({"file_path": file_path_with_table})

# 결과 확인
if result_with_table["has_table"]:
    print("Tables found:", len(result_with_table["table_data"]))
    print("\nTable HTML examples:")
    for table in result_with_table["table_data"][:2]:  # 처음 2개 테이블만 출력
        print(table["html"][:200] + "...")  # HTML의 처음 200자만 출력
        
print("\nExtracted content length:", len(result_with_table["content"]))

In [ ]:
# 테이블이 없는 PDF 파일 경로
file_path_without_table = "data/리비안_KR_without_table.pdf"

# 그래프 실행
result_without_table = graph.invoke({"file_path": file_path_without_table})

# 결과 확인
if result_without_table["has_table"]:
    print("Tables found:", len(result_without_table["table_data"]))
    print("\nTable HTML examples:")
    for table in result_without_table["table_data"][:2]:  # 처음 2개 테이블만 출력
        print(table["html"][:200] + "...")  # HTML의 처음 200자만 출력

print("\nExtracted content length:", len(result_without_table["content"]))